# Debugging repair, one stop at a time

This controlled circuit has two supplied linear writers that reconstruct the same source, followed by a threshold consumer. No weights or semantic roles are learned. We will delete the first writer's result, watch the second writer repair it, suppress that repair in a separate path, and replay a saved cut in a fresh process.

Install `saturn-pub[torch,notebooks]` and run the cells in order.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'examples/repair_circuit.py').exists(), 'Run from this public checkout'
sys.path.insert(0, str(ROOT / 'examples'))
from repair_circuit import RepairCircuit
from saturn_pub import Act, Session, StateAddress
from saturn_pub.cli import Debugger
from saturn_pub.paths import PathSchedule, TimedAct, Trajectory, compare_trajectories
from saturn_pub.symbols import SymbolBinding, SymbolTable
from saturn_pub.store import LocalStore
from saturn_pub.values import describe

session = RepairCircuit().session()
parent = session.capture()
native = Trajectory.run(session, name='native', parent=parent, steps=3, ports=('carrier', 'output'))
early = Trajectory.run(session, name='early', parent=parent, steps=3, ports=('carrier', 'output'),
    schedule=PathSchedule((TimedAct(1, Act.zero('carrier')),)))
both = Trajectory.run(session, name='both', parent=parent, steps=3, ports=('carrier', 'output'),
    schedule=PathSchedule((TimedAct(1, Act.zero('carrier')), TimedAct(2, Act.zero('carrier')))))

## Attach supplied, qualified symbols

Each binding names the first carrier channel at one exact execution point, with the source as a context dependency. The symbol table is optional read-only debug information. It grants no write authority.

In [2]:
bindings = []
for point in native.points:
    at_point = Session.from_cut(session.adapter, point.cut)
    bindings.append(SymbolBinding.bind(at_point, 'action.target',
        (StateAddress('carrier', (0,), role='carrier'),), context='supplied-left-contact-v1',
        context_slots=('source',), evidence=('repair_circuit.py: supplied role',),
        claim='First channel in supplied circuit; not learned.'))
debugger = Debugger(session)
debugger.symbols = SymbolTable(tuple(bindings))
debugger.symbol_context = 'supplied-left-contact-v1'
debugger.trajectories = {'native': native, 'early': early, 'both': both}
print(json.dumps(debugger.execute('info steps'), indent=2))
print(debugger.execute('resolve action.target')['status'])

{
  "point": {
    "family": "controlled-repair",
    "logical_step": 0,
    "phase": "execute",
    "operator": "writer.first",
    "index": 0,
    "edge": "before",
    "next_operation": "writer.first",
    "local_clock": 0
  },
  "next_transition": {
    "operation": "writer.first",
    "reads": [
      "source",
      "carrier"
    ],
    "writes": [
      "carrier",
      "cursor"
    ],
    "invalidates": [],
    "consumer": "two-channel-contact-threshold",
    "footprint": "declared"
  }
}
supplied


## Delete, then watch native repair

The same `Debugger.execute` parser powers the terminal. `continue` stops at a change watch and retains its safe-point cut. Step the first writer, delete the carrier, then continue into the second native writer.

In [3]:
debugger.execute('capture parent')
debugger.execute('fork early parent')
debugger.execute('use early')
debugger.execute('watch symbol:action.target change')
first = debugger.execute('continue')
assert first['stop'] == 'watchpoint'
print('After first writer:', debugger.session.read('carrier').tolist())
debugger.execute('zero carrier')
debugger.execute('capture deleted')
print('After deletion:', debugger.session.read('carrier').tolist())
repaired = debugger.execute('continue')
assert repaired['stop'] == 'watchpoint'
print('After native repair:', debugger.session.read('carrier').tolist())
print('Watch cut:', repaired['events'][0]['cut'])

After first writer: [1.0, 0.0]
After deletion: [0.0, 0.0]
After native repair: [1.0, 0.0]
Watch cut: 48db2db3fff513e33614494b363430360fdbfc97c5a0d9b68c65db39aab0659d


In [4]:
print(json.dumps(debugger.execute('backtrace carrier'), indent=2))
debugger.execute('unwatch all')
debugger.execute('step')
print('Native consumer:', debugger.execute('read output'))
assert debugger.execute('read output') == [1.0, 0.0]
comparison = debugger.execute('diff native early --first-divergence')
print('First recorded divergence:', comparison['first_recorded_divergence'])
print('Recovery windows:', comparison['observable_windows'])
print('Suppressed repair output:', both.points[-1].values['output'].tolist())
assert both.points[-1].values['output'].tolist() == [0.0, 0.0]

{
  "slot": "carrier",
  "writers": [
    {
      "receipt": "2ad67c8cd06203daa5087cd3b6b05f988ec7c546c5e8592ba17d66e5efedc0f8",
      "enclosing_receipt": null,
      "cut": "48db2db3fff513e33614494b363430360fdbfc97c5a0d9b68c65db39aab0659d",
      "cut_scope": "receipt-reference",
      "receipt_result": "48db2db3fff513e33614494b363430360fdbfc97c5a0d9b68c65db39aab0659d",
      "operation": "writer.repair",
      "writes_needed": [
        "carrier"
      ],
      "reads": [
        "source",
        "carrier"
      ],
      "point": {
        "edge": "before",
        "family": "controlled-repair",
        "index": 2,
        "local_clock": 2,
        "logical_step": 0,
        "next_operation": "consumer.threshold",
        "operator": "consumer.threshold",
        "phase": "execute"
      },
      "status": "declared-dependency"
    },
    {
      "receipt": "b7d106c4cfc300209ae47f6136d5adc24d59884a4a026be2084c3674237837b4",
      "enclosing_receipt": null,
      "cut": "45ad6956dcb

## Reopen the deleted state in two new processes

Durable debug information references the cut separately from its execution payload. Reconstruct the same supplied model, restore the cut, and test native repair or suppression after the next writer.

In [5]:
output = ROOT / 'outputs/notebooks/repair'
output.mkdir(parents=True, exist_ok=True)
store = LocalStore(output / 'store')
deleted = debugger.cuts['deleted']
store.save(deleted)
sidecar = output / 'debug-info.json'
debugger.symbols.save(sidecar, deleted)
for name, trace, suppress in [('repair', early, False), ('suppressed', both, True)]:
    command = [sys.executable, str(ROOT / 'examples/debugging_repair.py'), '--replay',
               str(output / 'store'), deleted.fingerprint, str(sidecar)]
    if suppress:
        command.append('--suppress')
    result = json.loads(subprocess.check_output(command, text=True))
    assert result['payload'] == describe(trace.points[-1].cut.payload)
    assert result['cut'] == trace.points[-1].cut.fingerprint
    print(name, result['metrics'], 'fresh-process complete state/cut: exact')
assert session.capture(retain=False).fingerprint == parent.fingerprint

repair {'target_contact': 1.0, 'collateral_contact': 0.0} fresh-process complete state/cut: exact


suppressed {'target_contact': 0.0, 'collateral_contact': 0.0} fresh-process complete state/cut: exact


The supplied circuit establishes a controlled repair mechanism: the first deletion is observable, the second writer restores the native carrier, and suppressing its result changes the consumer. In a pretrained model, an observable returning to native content alone is not a semantic repair claim. Qualify the symbol, native evaluator, writable boundary, and numerical contract separately.

Run `python examples/debugging_repair.py` for the full no-op, late-deletion, and collateral panel and a saved walkthrough.